# Notebook 07 — Colab E2E CV Launch (VS Code Extension)
## SentinelFatal2 — End-to-End 5-Fold Cross-Validation

**הרץ נוטבוק זה דרך תוסף Google Colab ב-VS Code עם Kernel מסוג T4 GPU.**

### איך לחבר:
1. פתח נוטבוק זה ב-VS Code
2. לחץ **Select Kernel** (פינה שמאלית עליונה) → **Colab** → **New Colab Server** → בחר T4 GPU
3. הרץ תאים 1–6 לפי הסדר

| תא | פעולה | זמן |
|----|--------|-----|
| 1 | **GPU CHECK** — אשר T4 לפני המשך | שניות |
| 2 | Clone repo לשרת Colab | ~1 דקה |
| 3 | הורדת נתונים מ-GitHub אם חסרים | ~1–3 דקות |
| 4 | התקנת תלויות | ~1 דקה |
| 5 | **Dry-run** — בדוק שהכל עובד על GPU | ~5 דקות |
| 6 | **ריצה מלאה** — חוסם עד סיום (3–5 שעות) | לילה שלם |
| 7 | בדיקת תוצאות בבוקר | שניות |

> ⚠️ **השאר VS Code פתוח כל הלילה.**
> דרך התוסף, הריצה חיה כל עוד VS Code מחובר לשרת.
> אם VS Code נסגר — הריצה נעצרת. אין בעיה: הסקריפט תומך ב-resume אוטומטי — פשוט הרץ שוב את Cell 6.

In [1]:
# ── Cell 1: GPU CHECK ────────────────────────────────────────────────────────
# NOTE: First run on a new Colab server takes ~60s to load PyTorch.
# This is normal — wait for the [*] to become a number.
print("Loading PyTorch... (first run takes ~60s)", flush=True)

import torch

has_gpu = torch.cuda.is_available()

print("=" * 50, flush=True)
print("  SentinelFatal2 — Hardware Check", flush=True)
print("=" * 50, flush=True)

if has_gpu:
    DEVICE   = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU  : {gpu_name}", flush=True)
    print(f"  VRAM : {gpu_mem:.1f} GB", flush=True)
    print(f"  CUDA : {torch.version.cuda}", flush=True)
    print(f"  GPU ready - safe to run Cell 2.", flush=True)
else:
    DEVICE = 'cpu'
    print("  NO GPU detected.", flush=True)
    print("  Fix: click kernel name (top-right) ->", flush=True)
    print("  'New Colab Server' -> T4 GPU -> reconnect.", flush=True)
    print("  Then re-run Cell 1.", flush=True)

print("=" * 50, flush=True)
print(f"  DEVICE = '{DEVICE}'", flush=True)
print("=" * 50, flush=True)

if not has_gpu:
    raise RuntimeError("No GPU. Switch to T4 GPU and re-run Cell 1.")


In [20]:
# ── Cell 2: Clone / Update Repository onto Colab Server ─────────────────────
# Although this notebook runs inside VS Code, the code executes on Google's
# remote server (/content/). We clone the repo there so all training scripts
# and config files are available on the remote machine.

import os, sys
from pathlib import Path

REPO_URL = "https://github.com/ArielShamay/SentinelFatal2.git"
REPO_DIR = Path("/content/SentinelFatal2")

if REPO_DIR.exists():
    print("Repository already on server — pulling latest changes...")
    os.chdir(REPO_DIR)
    print(os.popen("git pull").read())
else:
    print("Cloning repository to /content/SentinelFatal2 ...")
    ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed. Check internet connection.")
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")

# Add repo root to Python path so 'from src.xxx import ...' works
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("✅ Repository ready on Colab server.")

In [7]:
# ── Cell 3: Data Setup — Extract .npy Files from repo ZIP ───────────────────
# data_processed.zip is committed in the repo (~23 MB compressed).
# Cell 2 already cloned/pulled the repo, so the zip is at REPO_DIR/data_processed.zip.
# This cell extracts it into data/processed/ if the .npy files aren't already there.

import zipfile
from pathlib import Path

REPO_DIR  = Path("/content/SentinelFatal2")
ZIP_PATH  = REPO_DIR / "data_processed.zip"
DATA_DIR  = REPO_DIR / "data"

ctu_dir   = DATA_DIR / "processed" / "ctu_uhb"
fhrma_dir = DATA_DIR / "processed" / "fhrma"
ctu_npy   = list(ctu_dir.glob("*.npy"))   if ctu_dir.exists()   else []
fhrma_npy = list(fhrma_dir.glob("*.npy")) if fhrma_dir.exists() else []

if len(ctu_npy) >= 552 and len(fhrma_npy) >= 135:
    print("✅ Processed data already present — skipping extraction.")
    print(f"   ctu_uhb={len(ctu_npy)}, fhrma={len(fhrma_npy)}")
else:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f"ZIP not found at {ZIP_PATH}\n"
            "Make sure Cell 2 ran successfully (git clone/pull should include data_processed.zip)."
        )
    print(f"Extracting {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/1e6:.1f} MB) → {DATA_DIR} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print("Extraction complete.")

# ── Final verification ───────────────────────────────────────────────────────
ctu_npy   = list((DATA_DIR / "processed" / "ctu_uhb").glob("*.npy"))
fhrma_npy = list((DATA_DIR / "processed" / "fhrma").glob("*.npy"))

print(f"\nVerification:")
print(f"  ctu_uhb .npy : {len(ctu_npy)}  (expected 552)")
print(f"  fhrma   .npy : {len(fhrma_npy)}  (expected 135)")

for split in ['train', 'val', 'test']:
    p = REPO_DIR / "data" / "splits" / f"{split}.csv"
    status = '✅' if p.exists() else '❌ MISSING — re-run Cell 2 (git pull)'
    print(f"  {split}.csv      : {status}")

pt_ckpt = REPO_DIR / "checkpoints" / "pretrain" / "best_pretrain.pt"
ft_ckpt = REPO_DIR / "checkpoints" / "finetune" / "best_finetune.pt"
print(f"  best_pretrain.pt : {'✅' if pt_ckpt.exists() else '⚠  not found (OK for E2E CV)'}")
print(f"  best_finetune.pt : {'✅' if ft_ckpt.exists() else '⚠  not found (OK for E2E CV)'}")

if len(ctu_npy) < 552:
    raise RuntimeError(f"Expected 552 ctu_uhb .npy files, got {len(ctu_npy)}.")
if len(fhrma_npy) < 135:
    raise RuntimeError(f"Expected 135 fhrma .npy files, got {len(fhrma_npy)}.")

print("\n✅ Data ready — safe to run Cell 4.")


In [8]:
# ── Cell 4: Install Dependencies ─────────────────────────────────────────────
# Colab already has numpy, pandas, matplotlib, scipy, scikit-learn, torch.
# We only need to confirm PyYAML and tqdm are available.

import importlib, subprocess, sys

REQUIRED = {
    "yaml":         "PyYAML>=6.0",
    "tqdm":         "tqdm>=4.65",
    "sklearn":      "scikit-learn>=1.3",
}

to_install = []
for module, pkg in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        to_install.append(pkg)

if to_install:
    print(f"Installing: {to_install}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)
else:
    print("✅ All dependencies already installed.")

# Quick sanity imports
import torch, numpy as np, pandas as pd, yaml, sklearn
print(f"\nPackage versions:")
print(f"  torch      : {torch.__version__}  (CUDA: {torch.version.cuda})")
print(f"  numpy      : {np.__version__}")
print(f"  pandas     : {pd.__version__}")
print(f"  sklearn    : {sklearn.__version__}")
print(f"  PyYAML     : {yaml.__version__}")
print(f"\n  Device     : {DEVICE}")

In [21]:
# ── Cell 5: DRY-RUN — Verify full pipeline works on GPU (~2 min) ─────────────
# Runs 2 folds with max_batches=2 (StratifiedKFold requires n_splits>=2).
# --skip-pretrain: copies shared best_pretrain.pt instead of retraining (~35 min saved per fold).
# Output streams live line-by-line.
# If this cell fails, DO NOT proceed to Cell 6.

import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
os.chdir(REPO_DIR)

cmd = [
    sys.executable, "scripts/run_e2e_cv.py",
    "--device", DEVICE,
    "--dry-run",
    "--folds", "2",
    "--force-folds",
    "--skip-pretrain",
    "--seed", "42",
]

print("Starting dry-run (2 folds, max_batches=2, --skip-pretrain)...", flush=True)
print("Estimated time: ~2 minutes on T4 GPU", flush=True)
print("-" * 60, flush=True)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line.rstrip(), flush=True)
proc.wait()

print("-" * 60, flush=True)
if proc.returncode == 0:
    print("✅ Dry-run PASSED — safe to run Cell 6.", flush=True)
else:
    print(f"❌ Dry-run FAILED (exit code {proc.returncode}).", flush=True)
    print("   Fix errors above before running Cell 6.", flush=True)


In [ ]:
# ── Cell 6: FULL RUN (~60-90 min with stride=24) ────────────────────────────
# S14 Improvements vs previous run (AUC=0.565):
#   --skip-pretrain : use shared best_pretrain.pt (trained on all 687 recs)
#   lr_backbone=5e-5: 5× higher backbone LR (was 1e-5, backbone nearly frozen)
#   warmup=5 epochs : linear LR warmup to protect pretrained weights
#   EMA val_AUC     : smooth checkpointing (beta=0.8) reduces noisy early stop
#   StandardScaler  : feature normalization before LR (was raw features)
#   C=0.1           : tighter regularization (was 0.5)
#   stride=24       : 1 patch step (was 60), ~3× more windows for feature extraction
#   record-level features: n_alert_segments + alert_fraction (6 features total)
#
# Expected time: ~60-90 min on T4 GPU (stride=24 is ~2.5× slower than stride=60)
# ⚠️  Keep VS Code open. If it disconnects, re-run Cell 6 — auto-resumes.

import os, sys, subprocess, shutil
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
LOG_FILE = REPO_DIR / "logs" / "e2e_cv_master.log"
(REPO_DIR / "logs").mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

# ── Clear old fold artifacts so S14 improvements actually run ────────────────
for stale in [
    REPO_DIR / "checkpoints" / "e2e_cv",
    REPO_DIR / "results"     / "e2e_cv",
    REPO_DIR / "logs"        / "e2e_cv",
]:
    if stale.exists():
        shutil.rmtree(stale)
        print(f"  Cleared stale artifacts: {stale.name}/")
print("  Shared pretrain checkpoint preserved ✅")
print()

cmd = [
    sys.executable, "scripts/run_e2e_cv.py",
    "--device", DEVICE,
    "--force-folds",
    "--folds", "5",
    "--stride", "24",
    "--seed", "42",
    "--skip-pretrain",
]

print("=" * 60)
print("  SentinelFatal2 — Full 5-Fold E2E CV (S14 improvements)")
print(f"  Device        : {DEVICE}")
print("  skip-pretrain : shared best_pretrain.pt (687 recordings)")
print("  lr_backbone   : 5e-5 (was 1e-5)")
print("  warmup        : 5 epochs linear ramp")
print("  EMA beta      : 0.8 (smooth checkpointing)")
print("  StandardScaler: ON (before LR)")
print("  LR C          : 0.1 (was 0.5)")
print("  stride        : 24 (1 patch step, was 60)")
print("  features      : 6 (4 segment + 2 record-level)")
print("  Expected time : ~60-90 min on T4 GPU")
print("=" * 60)
print()

log_lines = []
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    line_stripped = line.rstrip()
    print(line_stripped, flush=True)
    log_lines.append(line)
proc.wait()

with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.writelines(log_lines)

print()
print("=" * 60)
if proc.returncode == 0:
    print("✅ ALL FOLDS COMPLETE — run Cell 7 to see final results.")
else:
    print(f"⚠️  Finished with exit code {proc.returncode}.")
    print("   Some folds may have failed. Run Cell 7 to check partial results.")
print("=" * 60)

In [ ]:
# ── Cell 7: MORNING CHECK — Read Results ─────────────────────────────────────
# Run this cell in the morning to check on the run.
# Can be run safely at any time — it only reads files, never modifies anything.

import os, pandas as pd
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
LOG_FILE = REPO_DIR / "logs" / "e2e_cv_master.log"
PROGRESS = REPO_DIR / "logs" / "e2e_cv_progress.csv"
REPORT   = REPO_DIR / "results" / "e2e_cv_final_report.csv"

print("=" * 60)
print("  SentinelFatal2 — E2E CV Morning Check")
print("=" * 60)

# Is the process still running?
pid_check = os.popen("pgrep -f run_e2e_cv.py").read().strip()
if pid_check:
    print(f"\n  Process still running (PID: {pid_check})")
else:
    print("\n  Process has finished (or was interrupted).")

# Last 40 lines of log
print("\n-- Last 40 lines of log " + "-" * 36)
os.system(f"tail -40 {LOG_FILE}")

# Per-fold progress
print("\n-- Per-fold progress " + "-" * 39)
if PROGRESS.exists():
    df_prog = pd.read_csv(PROGRESS)
    print(df_prog.to_string(index=False))
else:
    print("  (no progress file yet — run may still be in fold 0)")

# Final report
print("\n-- Final report " + "-" * 44)
if REPORT.exists():
    df_rep = pd.read_csv(REPORT)
    print(df_rep.to_string(index=False))
    print("\n✅ Run COMPLETE.")
else:
    print("  (no final report yet — run has not completed all 5 folds)")

print("\n-- ROC curve plot " + "-" * 42)
img_path = REPO_DIR / "docs" / "images" / "e2e_cv.png"
if img_path.exists():
    from IPython.display import Image, display
    display(Image(str(img_path)))
else:
    print("  (plot not yet generated)")
print("=" * 60)

In [ ]:
import subprocess, os
from pathlib import Path
REPO_DIR = Path("/content/SentinelFatal2")

# Show NEW log and results
NEW_LOG  = REPO_DIR / "logs" / "e2e_cv_master.log"
NEW_PROG = REPO_DIR / "logs" / "e2e_cv_progress.csv"
NEW_REP  = REPO_DIR / "results" / "e2e_cv_final_report.csv"

print(f"Log exists: {NEW_LOG.exists()}, size: {NEW_LOG.stat().st_size if NEW_LOG.exists() else 0}")
print()
print("=== Last 20 lines of new log ===")
out = subprocess.run(["tail", "-20", str(NEW_LOG)], capture_output=True, text=True)
print(out.stdout or "(empty)")

print("=== Progress ===")
print(NEW_PROG.read_text() if NEW_PROG.exists() else "not found")

print("=== Final Report ===")
print(NEW_REP.read_text() if NEW_REP.exists() else "not found")

# Also check new fold checkpoints exist
print("=== New fold checkpoints ===")
for k in range(5):
    pt = REPO_DIR / "checkpoints" / "e2e_cv" / f"fold{k}" / "finetune" / "best_finetune.pt"
    oof = REPO_DIR / "results" / "e2e_cv" / f"fold{k}_oof_scores.csv"
    print(f"  fold{k}: finetune.pt={'✅' if pt.exists() else '❌'}, oof={'✅' if oof.exists() else '❌'}")
